# Assignment 06: Clinic Data Wrangling Pipeline (Midterm)

Follow `README.md`: it names every file to save in `output/`, with its first line and line count. Replace each **TODO**, then restart the kernel and **Run All** before you commit, so the saved outputs show your work.

In [1]:
# Supplied setup: run this cell first and leave it unchanged.
from pathlib import Path
import hashlib
import json
import subprocess
import sys

import numpy as np
import pandas as pd

print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| pandas", pd.__version__)


def find_assignment_dir(start):
    """The folder holding data_manifest.json: this one, a parent, or 06/mid-term below one."""
    for folder in (start, *start.parents):
        for candidate in (folder, folder / "06" / "mid-term"):
            if (candidate / "data_manifest.json").is_file():
                return candidate
    raise FileNotFoundError("Open this notebook from the assignment folder that holds data_manifest.json.")


BASE_DIR = find_assignment_dir(Path.cwd().resolve())
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)
PROFILE_PATH = OUTPUT_DIR / "table_profile.csv"
AUDIT_PATH = OUTPUT_DIR / "issue_audit.csv"
PATIENTS_SUMMARY_PATH = OUTPUT_DIR / "patients_summary.csv"
GENDER_CATEGORIES_PATH = OUTPUT_DIR / "gender_categories.csv"
VISITS_SUMMARY_PATH = OUTPUT_DIR / "visits_summary.csv"
DECISION_LOG_PATH = OUTPUT_DIR / "decision_log.csv"
A1C_SUMMARY_PATH = OUTPUT_DIR / "a1c_summary.csv"
MERGE_AUDIT_PATH = OUTPUT_DIR / "merge_audit.csv"
SBP_WIDE_SAMPLE_PATH = OUTPUT_DIR / "sbp_wide_sample.csv"
ANALYSIS_SAMPLE_PATH = OUTPUT_DIR / "analysis_sample.csv"

# The patients whose rows the two extracts hold (README Tasks 4.3 and 5.2).
SAMPLE_IDS = ["P00001", "P00004", "P00088", "P00260", "P00446", "P00487", "P00631", "P00851",
              "P02354", "P03836", "P05763", "P07510", "P14287", "P16486", "P17390", "P30001"]

# The data is generated, not committed: generate_data.py writes the same tables on every computer.
manifest = json.loads((BASE_DIR / "data_manifest.json").read_text(encoding="utf-8"))
if not all((DATA_DIR / name).is_file() for name in manifest["files"]):
    print(subprocess.run([sys.executable, "generate_data.py"], cwd=BASE_DIR, check=True,
                         capture_output=True, text=True).stdout.strip())


def read_raw(path):
    """A Parquet file with the dtypes it stores; a CSV file with every column as text, blanks kept as written."""
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path, dtype="string", keep_default_na=False)


raw = {Path(name).stem: read_raw(DATA_DIR / name) for name in manifest["files"]}
# The checksum covers each table's contents, written out as CSV text.
for name, expected in manifest["files"].items():
    text = raw[Path(name).stem].to_csv(index=False, lineterminator="\n")
    assert hashlib.sha256(text.encode("utf-8")).hexdigest() == expected["sha256"], (
        f"data/{name} differs from the generated file; delete the data folder and run this cell again.")
print(f"Verified {len(manifest['files'])} data files")
raw_snapshot = {name: table.copy(deep=True) for name, table in raw.items()}
list(raw)

Python 3.13.15 | NumPy 2.3.3 | pandas 3.0.5
Verified 5 data files


['patients', 'sites', 'visits_north', 'visits_south', 'a1c_wide']

## Cumulative midterm checkpoint (human review)

**TODO:** Replace this text with an evidence map: one bullet for each of Lectures 01 to 06. For 01, 02, 04, 05, and 06, name one concrete technique or file and say in one sentence where this notebook or its outputs use it. For 03, complete the NumPy calculation after Task 4.3 and explain your axis choice in one sentence here. The lectures cover the command line and Python basics (01); Git, functions, and files (02); environments, shell pipelines, and NumPy (03); Jupyter and labeled pandas data (04); data contracts, cleaning, and validation (05); and joins, reshaping, and concatenation (06). Do not paste earlier solutions.

Lecture 01-  I used the command line in Task 1.1 with wc -l data/*.csv > output/file_inventory.txt to count the lines in the CSV files.

Lecture 02 —  I created a profile_table() function to calculate the rows, columns, and exact duplicate rows for each input table and saved the results to output/table_profile.csv.

Lecture 03 — I used axis=0 so I can have one mean per each visit (rows are patients)

Lecture 04 — I used pandas DataFrames to label colums for later cleaning. 

Lecture 05 — Data contracts, cleaning, and validation: In Tasks 2 & 3, I defined the intended keys and grain of the tables, normalized gender and site codes, converted invalid values to missing values, and validated blood pressure values were in the right range(70-250)

Lecture 06 — Joins, reshaping, and concatenation: I used pd.concat() to stack the north and south visit exports, and merges with validate= to check expected relationships. 



## Task 1: Inventory the files

### 1.1 Count lines from the terminal

Make `output/file_inventory.txt` in the terminal (README Task 1.1), then paste the command you ran into the `%%bash` cell below.

In [2]:
%%bash
# paste your command here
wc -l data/*.csv > output/file_inventory.txt

### 1.2 Profile each table with a function

In [3]:
# TODO: write a function, such as profile(name, table), that returns the four values in README
# Task 1.2, call it once per name in manifest["files"] with raw[Path(name).stem], and save the results to PROFILE_PATH.
def profile_table(name, table):
    return {
        "table": name,
        "rows": len(table),
        "columns": table.shape[1],
        "exact_duplicate_rows": table.duplicated().sum()
    }


results = []

for name in manifest["files"]:
    table = raw[Path(name).stem]
    results.append(profile_table(name, table))


table_profile = pd.DataFrame(results)

table_profile.to_csv(PROFILE_PATH, index=False)

table_profile


,table,rows,columns,exact_duplicate_rows
0,patients.parquet,30045,4,45
1,sites.csv,5,3,0
2,visits_north.csv,26297,6,35
3,visits_south.csv,31074,6,35
4,a1c_wide.parquet,29424,3,0


## Task 2: Contract, stack, and audit

### 2.1 Write the data contract (human review)

**TODO:** Replace this text with the contract: the intended grain and key (single or composite) of each cleaned input table, and the cardinality you expect for each Task 4.4 join. Describe visits after stacking and keeping the later recheck, sites after keeping current records, and the long-form A1c join keys. Note raw exact repeats, rechecks, and site versions that violate the intended cleaned keys (README Task 2.1).

- In patients.parquet the grain was a registered patient and the key was "patient_id". Duplicates will be removed during cleaning. 


- In sites.csv the grain was a site record and the key was "site_code". Only the current records will be kept after cleaning.  


- In visits_north.csv and visits_south.csv the grain is the classification of visit (baseline or week12) and the Key is is the key patient_id + visit. Duplicates will be removed during cleaing and rechecks will be kept.

In a1c_wide.parquet the grain is a patient's central lab results and the key is patient_id. 

Cardinalities: 
visits to patients: many to one; a patient can have multiple visits

patients to sites: many_to_one; many patients can be at one site. 

Visits to long-form A1c results: one_to_one; each visit can only have one long-gorm A1c result

### 2.2 Stack the two visit files

In [ ]:
# TODO: rename wt_kg, add source_file to each visit table, and stack them, north first (README Task 2.2).
north = raw["visits_north"].copy(deep=True)
south = raw["visits_south"].copy(deep=True)
south = south.rename(
    columns={"wt_kg": "weight_kg"})
north["source_file"] = "visits_north.csv"
south["source_file"] = "visits_south.csv"
visits_raw = pd.concat(
    [north, south],
    ignore_index=True
)



(57371, 7)


,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
0,P00002,baseline,2026-02-20,09:50,132,85.2,visits_north.csv
1,P00002,week12,2026-05-08,15:20,129,82.0,visits_north.csv
2,P00003,baseline,2026-01-16,11:45,148,67.5,visits_north.csv
3,P00003,wk12,2026-04-07,10:00,137,64.0,visits_north.csv
4,P00004,baseline,2026-02-25,10:55,145,106.2,visits_north.csv


### 2.3 Audit the raw tables

In [21]:
visits_raw["visit"].value_counts(dropna=False)

visit
baseline     28533
week12       26529
BASELINE       256
Baseline       242
bl             235
baseline       228
BL             220
wk12           204
 week12        198
Week 12        195
WEEK12         186
Wk12           177
week 12        168
Name: count, dtype: int64[pyarrow]

In [23]:
# TODO: count the 10 issues in README Task 2.3, in order, as a DataFrame with the columns issue and
# count, and save it to AUDIT_PATH.

patients = raw["patients"]
sites = raw["sites"]


# gender values that are not a code
gender_stripped = patients["gender"].str.strip()

gender_not_code = (
    (gender_stripped != "")
    & (~patients["gender"].isin(["F", "M", "X"]))
).sum()


# empty gender
empty_gender = (gender_stripped == "").sum()


# birth_year sentinel 9999
birth_year_9999 = (
    patients["birth_year"] == "9999"
).sum()


# site codes needing normalization
site_normalized = (
    patients["site_code"]
    .str.upper()
    .str.replace("-", "", regex=False)
    .str.replace(" ", "", regex=False)
)

site_needs_normalization = (
    patients["site_code"] != site_normalized
).sum()


#rows with a repeated site_code
repeated_site_code = sites.duplicated(
    subset=["site_code"],
    keep=False
).sum()


#visit labels that are not baseline or week12
bad_visit_labels = (
    ~visits_raw["visit"].isin(["baseline", "week12"])
).sum()


# sbp sentinel 999
sbp_numeric = pd.to_numeric(
    visits_raw["sbp_mmhg"],
    errors="coerce"
)


# 7. Visits: sbp sentinel 999
sbp_999 = (
    sbp_numeric == 999
).sum()


# sbp outside 70 through 250
sbp_outside = (
    (sbp_numeric != 999)
    & (
        (sbp_numeric < 70)
        | (sbp_numeric > 250)
    )
).sum()


# date parse failures
date_text = visits_raw["visit_date"]

parsed_dates = pd.to_datetime(
    date_text,
    format="%Y-%m-%d",
    errors="coerce"
)

exact_date = (
    date_text.str.fullmatch(r"\d{4}-\d{2}-\d{2}")
    & parsed_dates.notna()
)

date_parse_failures = (
    ~exact_date
).sum()


# rows in repeated patient_id and visit pairs
visit_normalized = (
    visits_raw["visit"]
    .str.strip()
    .str.lower()
)

# Map the actual spellings found in value_counts()
# to the two valid visit codes.
visit_map = {
    "baseline": "baseline",
    "bl": "baseline",
    "week12": "week12",
    "week 12": "week12",
    "wk12": "week12",
}

visit_for_audit = visit_normalized.map(visit_map)


visit_pairs = visits_raw[["patient_id"]].copy()
visit_pairs["visit"] = visit_for_audit


repeated_visit_pairs = visit_pairs.duplicated(
    subset=["patient_id", "visit"],
    keep=False
).sum()



issue_audit = pd.DataFrame({
    "issue": [
        "patients: gender values that are not a code",
        "patients: empty gender",
        "patients: birth_year sentinel 9999",
        "patients: site codes needing normalization",
        "sites: rows with a repeated site_code",
        "visits: visit labels that are not baseline or week12",
        "visits: sbp sentinel 999",
        "visits: sbp outside 70 through 250",
        "visits: date parse failures",
        "visits: rows in repeated patient_id and visit pairs",
    ],
    "count": [
        gender_not_code,
        empty_gender,
        birth_year_9999,
        site_needs_normalization,
        repeated_site_code,
        bad_visit_labels,
        sbp_999,
        sbp_outside,
        date_parse_failures,
        repeated_visit_pairs,
    ],
})


# Save the audit table.
issue_audit.to_csv(AUDIT_PATH, index=False)





assert all(raw[name].equals(raw_snapshot[name]) for name in raw), "Auditing must not change raw."

## Task 3: Clean

### 3.1 Clean the patients

In [25]:
# TODO: list the gender spellings with value_counts(), then build patients_clean from
# raw["patients"].copy(deep=True) with steps 1 to 5 in README Task 3.1.
patients_clean = raw["patients"].copy(deep=True)

patients_clean = patients_clean.drop_duplicates(
    keep="first"
)

patients_clean["gender"].value_counts(dropna=False)

gender_normalized = (
    patients_clean["gender"]
    .str.strip()
    .str.lower()
)

gender_map = {
    "f": "F",
    "female": "F",
    "woman": "F",
    "m": "M",
    "male": "M",
    "man": "M",
    "x": "X",
    "nonbinary": "X",
    "non-binary": "X",
    "nb": "X",
}

patients_clean["gender"] = (
    gender_normalized
    .map(gender_map)
    .astype("category")
)



patients_clean["birth_year"] = (
    pd.to_numeric(
        patients_clean["birth_year"].replace("9999", pd.NA),
        errors="coerce"
    )
    .astype("Int64")
)



patients_clean["site_code"] = (
    patients_clean["site_code"]
    .str.upper()
    .str.replace("-", "", regex=False)
    .str.replace(" ", "", regex=False)
)



patients_clean = patients_clean.reset_index(drop=True)


# Self-check: gender should have exactly F, M, X as categories
patients_clean["gender"].cat.categories

Index(['F', 'M', 'X'], dtype='str')

In [26]:
# TODO: show patients_clean["gender"].cat.categories, save the patients summary to
# PATIENTS_SUMMARY_PATH, and save the gender value_counts(dropna=False) with a code column from
# .cat.codes to GENDER_CATEGORIES_PATH, as in README Task 3.1.

print(patients_clean["gender"].cat.categories)


# Create summary
patients_summary = pd.DataFrame({
    "metric": [
        "patients",
        "birth_year missing",
        "birth_year min",
        "birth_year max",
        "site_code values",
        "patients at N01",
        "patients at S02",
        "patients at E04",
    ],
    "value": [
        len(patients_clean),
        patients_clean["birth_year"].isna().sum(),
        patients_clean["birth_year"].min(),
        patients_clean["birth_year"].max(),
        patients_clean["site_code"].nunique(),
        (patients_clean["site_code"] == "N01").sum(),
        (patients_clean["site_code"] == "S02").sum(),
        (patients_clean["site_code"] == "E04").sum(),
    ],
})


# Save summary
patients_summary.to_csv(
    PATIENTS_SUMMARY_PATH,
    index=False
)


# Create gender category counts
gender_counts = (
    patients_clean["gender"]
    .value_counts(dropna=False)
    .reset_index()
)


gender_counts["code"] = gender_counts["gender"].cat.codes


# Save counts
gender_counts.to_csv(
    GENDER_CATEGORIES_PATH,
    index=False
)
patients_summary

Index(['F', 'M', 'X'], dtype='str')


,metric,value
0,patients,30000
1,birth_year missing,371
2,birth_year min,1941
3,birth_year max,1996
4,site_code values,3
5,patients at N01,13739
6,patients at S02,15706
7,patients at E04,555


### 3.2 Clean the visits

In [34]:
# TODO: build visits_clean from visits_raw.copy(deep=True) with steps 1 to 5 in README Task 3.2.
visits_clean = visits_raw.copy(deep=True)


# Step 1
visits_clean = visits_clean.drop_duplicates(
    keep="first"
)


# Step 2
visits_clean["visit"].value_counts(dropna=False)

visit_normalized = (
    visits_clean["visit"]
    .str.strip()
    .str.lower()
)

visit_map = {
    "baseline": "baseline",
    "bl": "baseline",
    "week12": "week12",
    "wk12": "week12",
    "week 12": "week12",
}

visits_clean["visit"] = visit_normalized.map(visit_map)


# Step 3
sbp_numeric = pd.to_numeric(
    visits_clean["sbp_mmhg"],
    errors="coerce"
)

visits_clean["sbp_mmhg"] = (
    sbp_numeric
    .where(sbp_numeric.between(70, 250, inclusive="both"))
    .astype("Int64")
)


# Step 4
visits_clean["weight_kg"] = pd.to_numeric(
    visits_clean["weight_kg"],
    errors="coerce"
)


# Step 5
date_text = visits_clean["visit_date"]

parsed_dates = pd.to_datetime(
    date_text,
    format="%Y-%m-%d",
    errors="coerce"
)

exact_date = (
    date_text.str.fullmatch(r"\d{4}-\d{2}-\d{2}")
    & parsed_dates.notna()
)

visits_clean["visit_date"] = parsed_dates.where(exact_date)

In [35]:
# TODO: step 6, show the repeated patient_id and visit pairs, then keep the later reading.
repeated_pairs = visits_clean.duplicated(
    subset=["patient_id", "visit"],
    keep=False
)

# Show the repeated pairs
display(
    visits_clean.loc[repeated_pairs]
    .sort_values(["patient_id", "visit", "visit_time"])
)

# Sort by visit_time so the later reading comes last,
# then keep the later reading for each patient_id + visit pair
visits_clean = (
    visits_clean
    .sort_values("visit_time")
    .drop_duplicates(
        subset=["patient_id", "visit"],
        keep="last"
    )
)

,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
26298,P00001,baseline,2026-03-06,14:15,141,52.4,visits_south.csv
26297,P00001,baseline,2026-03-06,14:40,146,52.4,visits_south.csv
4,P00004,baseline,2026-02-25,10:55,145,106.2,visits_north.csv
5,P00004,baseline,2026-02-25,11:15,142,106.2,visits_north.csv
84,P00088,baseline,2026-03-10,09:30,153,68.2,visits_north.csv
...,...,...,...,...,...,...,...
55473,P28232,baseline,2026-01-12,10:10,159,61.0,visits_south.csv
56249,P28953,baseline,2026-01-21,10:30,153,74.8,visits_south.csv
56248,P28953,baseline,2026-01-21,10:45,151,74.8,visits_south.csv
25464,P29170,baseline,2026-03-02,11:20,130,78.0,visits_north.csv


In [37]:
# TODO: step 7, sort, renumber, keep the six columns, then save the visits summary to
# VISITS_SUMMARY_PATH (README Task 3.2).
visits_clean = (
  visits_clean
    .sort_values(["patient_id", "visit"])
    .reset_index(drop=True)
)


visits_clean = visits_clean[
    [
        "patient_id",
        "visit",
        "visit_date",
        "sbp_mmhg",
        "weight_kg",
        "source_file",
    ]
]


# Build summary
visits_summary = pd.DataFrame({
    "metric": [
        "visits",
        "baseline visits",
        "week12 visits",
        "visit_date missing",
        "visit_date min",
        "visit_date max",
        "sbp_mmhg missing",
        "sbp_mmhg min",
        "sbp_mmhg max",
        "weight_kg missing",
        "weight_kg mean",
        "visits from visits_north.csv",
        "visits from visits_south.csv",
    ],
    "value": [
        len(visits_clean),
        (visits_clean["visit"] == "baseline").sum(),
        (visits_clean["visit"] == "week12").sum(),
        visits_clean["visit_date"].isna().sum(),
        visits_clean["visit_date"].min(),
        visits_clean["visit_date"].max(),
        visits_clean["sbp_mmhg"].isna().sum(),
        visits_clean["sbp_mmhg"].min(),
        visits_clean["sbp_mmhg"].max(),
        visits_clean["weight_kg"].isna().sum(),
        round(visits_clean["weight_kg"].mean(), 2),
        (visits_clean["source_file"] == "visits_north.csv").sum(),
        (visits_clean["source_file"] == "visits_south.csv").sum(),
    ],
})


# Save summary
visits_summary.to_csv(
    VISITS_SUMMARY_PATH,
    index=False
)

visits_summary

,metric,value
0,visits,57181
1,baseline visits,29599
2,week12 visits,27582
3,visit_date missing,169
4,visit_date min,2026-01-05 00:00:00
5,visit_date max,2026-07-06 00:00:00
6,sbp_mmhg missing,456
7,sbp_mmhg min,70
8,sbp_mmhg max,250
9,weight_kg missing,828


### 3.3 Record the decisions

In [39]:
# TODO: one row per issue in README Task 3.3, with table and issue copied, and your own
# decision and one-sentence reason, then save it to DECISION_LOG_PATH.
decisions = [
    [
        "patients",
        "exact duplicate rows",
        "Remove exact duplicate rows and keep the first.",
        "Exact duplicates would count the same record multiple times."
    ],
    [
        "patients",
        "gender spellings that are not a code",
        "Strip and lowercase values, map known spellings to F, M, or X, and leave unknown or empty values missing.",
        "The registry contains different spellings of the same gender categories, while getting rid of unknown/empty values"
    ],
    [
        "patients",
        "site codes needing normalization",
        "Uppercase site codes and remove spaces and hyphens.",
        "Standardization"
    ],
    [
        "patients",
        "birth_year sentinel 9999",
        "Replace 9999 with a missing value and convert birth_year to Int64.",
        "Not real year"
    ],
    [
        "visits",
        "weight column named wt_kg in visits_south.csv",
        "Rename wt_kg to weight_kg before stacking the visit files.",
        "They both need the same name"
    ],
    [
        "visits",
        "sbp sentinel 999 and readings outside 70 through 250",
        "Convert SBP to numeric and make 999 and values outside 70 through 250 missing.",
        "Outside of allowed range"
    ],
    [
        "visits",
        "impossible and loosely written visit dates",
        "Keep only exact YYYY-MM-DD values that are valid calendar dates.",
        "Gets rid of any date not formatted correctly."
    ],
    [
        "visits",
        "repeated patient_id and visit pairs",
        "Sort rechecks by visit_time and keep the later reading.",
        "retain later readings"
    ],
    [
        "sites",
        "repeated site_code rows",
        "Keep only the current record for each site_code.",
        "Keeping current records provides one current site record per site_code."
    ],
]
decision_table = pd.DataFrame(
    decisions,
    columns=["table", "issue", "decision", "reason"]
)

decision_table.to_csv(DECISION_LOG_PATH, index=False)

decision_table

,table,issue,decision,reason
0,patients,exact duplicate rows,Remove exact duplicate rows and keep the first.,Exact duplicates would count the same record m...
1,patients,gender spellings that are not a code,"Strip and lowercase values, map known spelling...",The registry contains different spellings of t...
2,patients,site codes needing normalization,Uppercase site codes and remove spaces and hyp...,Standardization
3,patients,birth_year sentinel 9999,Replace 9999 with a missing value and convert ...,Not real year
4,visits,weight column named wt_kg in visits_south.csv,Rename wt_kg to weight_kg before stacking the ...,They both need the same name
5,visits,sbp sentinel 999 and readings outside 70 throu...,Convert SBP to numeric and make 999 and values...,Outside of allowed range
6,visits,impossible and loosely written visit dates,Keep only exact YYYY-MM-DD values that are val...,Gets rid of any date not formatted correctly.
7,visits,repeated patient_id and visit pairs,Sort rechecks by visit_time and keep the later...,retain later readings
8,sites,repeated site_code rows,Keep only the current record for each site_code.,Keeping current records provides one current s...


## Task 4: Reshape and join

### 4.1 Melt the A1c results to long form

In [41]:
# TODO: melt raw["a1c_wide"], map the visit labels, drop the rows with no result,
# then save the per-visit summary to A1C_SUMMARY_PATH (README Task 4.1).
a1c_long = raw["a1c_wide"].melt(
    id_vars=["patient_id"],
    value_vars=["a1c_baseline", "a1c_week12"],
    var_name="visit",
    value_name="a1c_pct"
)

a1c_long["visit"] = a1c_long["visit"].map({
    "a1c_baseline": "baseline",
    "a1c_week12": "week12",
})


print(a1c_long["a1c_pct"].dtype)

# Remove rows with no result (rmbr to finish)
a1c_long = (
    a1c_long
    .dropna(subset=["a1c_pct"])
    .reset_index(drop=True)
)

a1c_summary = pd.DataFrame({
    "visit": [
        "baseline",
        "week12",
    ],
    "results": [
        (a1c_long["visit"] == "baseline").sum(),
        (a1c_long["visit"] == "week12").sum(),
    ],
    "a1c_pct_mean": [
        round(
            a1c_long.loc[
                a1c_long["visit"] == "baseline",
                "a1c_pct"
            ].mean(),
            2
        ),
        round(
            a1c_long.loc[
                a1c_long["visit"] == "week12",
                "a1c_pct"
            ].mean(),
            2
        ),
    ],
})

a1c_summary.to_csv(
    A1C_SUMMARY_PATH,
    index=False
)

a1c_summary

float64


,visit,results,a1c_pct_mean
0,baseline,29107,8.15
1,week12,27653,7.63


### 4.2 Check the keys before joining

In [42]:
# TODO: steps 1 and 2, catch the MergeError, show the repeated site rows, and build sites_current.
sites = raw["sites"]


# Step 1
try:
    patients_clean.merge(
        sites,
        on="site_code",
        how="left",
        validate="many_to_one"
    )
except pd.errors.MergeError as error:
    print(error)


# Step 2
repeated_sites = sites.duplicated(
    subset=["site_code"],
    keep=False
)

display(
    sites.loc[repeated_sites]
)


# Keep current site records
sites_current = sites.loc[
    sites["record_status"] == "current",
    ["site_code", "site_name"]
].copy()


sites_current

Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 site_code
      N01
      S02 ...


,site_code,site_name,record_status
0,N01,North Clinic,current
1,N01,North Street Clinic,retired
2,S02,Southgate Clinic,retired
3,S02,South Valley Clinic,current


,site_code,site_name
0,N01,North Clinic
3,S02,South Valley Clinic
4,W03,West Clinic


In [43]:
# TODO: step 3, outer merge visits_clean and a1c_long with indicator=True and show the _merge counts,
# then save the seven counts in README Task 4.2 to MERGE_AUDIT_PATH.

visit_a1c_merge = visits_clean.merge(
    a1c_long,
    on=["patient_id", "visit"],
    how="outer",
    indicator=True
)

display(
    visit_a1c_merge["_merge"].value_counts()
)


visits_with_a1c = (
    visit_a1c_merge["_merge"] == "both"
).sum()


visits_without_a1c = (
    visit_a1c_merge["_merge"] == "left_only"
).sum()


a1c_without_visit = (
    visit_a1c_merge["_merge"] == "right_only"
).sum()


visits_patient_not_registry = (
    ~visits_clean["patient_id"].isin(
        patients_clean["patient_id"]
    )
).sum()


registry_patients_no_visit = (
    ~patients_clean["patient_id"].isin(
        visits_clean["patient_id"]
    )
).sum()

registry_site_not_list = (
    ~patients_clean["site_code"].isin(
        sites["site_code"]
    )
).sum()

site_rows_removed = (
    len(sites) - len(sites_current)
)


# Build merge audit
merge_audit = pd.DataFrame({
    "check": [
        "visits with an A1c result",
        "visits without an A1c result",
        "A1c results without a visit",
        "visits whose patient is not in the registry",
        "registry patients with no visit",
        "registry patients whose site is not in the site list",
        "site list rows removed by the current-record rule",
    ],
    "count": [
        visits_with_a1c,
        visits_without_a1c,
        a1c_without_visit,
        visits_patient_not_registry,
        registry_patients_no_visit,
        registry_site_not_list,
        site_rows_removed,
    ],
})


# Save audit
merge_audit.to_csv(
    MERGE_AUDIT_PATH,
    index=False
)

merge_audit

_merge
both          54006
left_only      3175
right_only     2754
Name: count, dtype: int64

,check,count
0,visits with an A1c result,54006
1,visits without an A1c result,3175
2,A1c results without a visit,2754
3,visits whose patient is not in the registry,79
4,registry patients with no visit,441
5,registry patients whose site is not in the sit...,555
6,site list rows removed by the current-record rule,2


### 4.3 Pivot blood pressure to one row per patient

In [44]:
# TODO: pivot visits_clean, reset the index, clear columns.name, add sbp_change, then save the
# SAMPLE_IDS rows to SBP_WIDE_SAMPLE_PATH (README Task 4.3).
sbp_wide = visits_clean.pivot(
    index="patient_id",
    columns="visit",
    values="sbp_mmhg"
)

sbp_wide = sbp_wide.reset_index()

sbp_wide.columns.name = None

# Calculate change in blood pressure
# week12 minus baseline
sbp_wide["sbp_change"] = (
    sbp_wide["week12"] - sbp_wide["baseline"]
)

sbp_wide_sample = sbp_wide[
    sbp_wide["patient_id"].isin(SAMPLE_IDS)
]

# Save sample
sbp_wide_sample.to_csv(
    SBP_WIDE_SAMPLE_PATH,
    index=False
)
sbp_wide_sample

,patient_id,baseline,week12,sbp_change
0,P00001,146,129,-17
3,P00004,142,<NA>,<NA>
87,P00088,152,147,-5
259,P00260,70,138,68
445,P00446,250,144,-106
486,P00487,<NA>,143,<NA>
849,P00851,<NA>,129,<NA>
2336,P02354,127,125,-2
3789,P03836,134,125,-9
5684,P05763,161,151,-10


#### NumPy checkpoint (Lecture 03, human review)

From the full `sbp_wide`, select `baseline` and `week12` in that order, keeping only patients with both readings. Name this two-column table `complete_pairs`; `np.array(complete_pairs, dtype=float)` converts it to a NumPy array. Display its shape and calculate one NumPy mean per visit across patients. Save the outputs and explain your axis choice in the cumulative checkpoint's Lecture 03 bullet. No extra output file.

In [45]:
# TODO: select complete pairs, convert to a NumPy array, and display its shape and two per-visit means.
complete_pairs = (
    sbp_wide[["baseline", "week12"]]
    .dropna()
)

sbp_array = np.array(
    complete_pairs,
    dtype=float
)

print(sbp_array.shape)

print(
    np.mean(sbp_array, axis=0)
)

(27149, 2)
[143.01337066 137.15061328]


### 4.4 Join the analysis table

In [47]:
# TODO: three left merges onto visits_clean, each with validate=, then the columns and dtypes
# in README Task 4.4. A MergeError means an earlier step left a problem: see HINTS.md.
analysis = visits_clean.copy()


# 1. Add patient info
analysis = analysis.merge(
    patients_clean,
    on="patient_id",
    how="left",
    validate="many_to_one"
)


# 2. Add current site info
analysis = analysis.merge(
    sites_current,
    on="site_code",
    how="left",
    validate="many_to_one"
)


# 3. Addresults
analysis = analysis.merge(
    a1c_long,
    on=["patient_id", "visit"],
    how="left",
    validate="one_to_one"
)


# set order
analysis = analysis[
    [
        "patient_id",
        "visit",
        "visit_date",
        "sbp_mmhg",
        "weight_kg",
        "a1c_pct",
        "gender",
        "birth_year",
        "site_code",
        "site_name",
        "source_file",
    ]
]


# set types
analysis = analysis.astype({
    "patient_id": "string",
    "visit": "string",
    "visit_date": "datetime64[us]",
    "sbp_mmhg": "Int64",
    "weight_kg": "float64",
    "a1c_pct": "float64",
    "gender": "string",
    "birth_year": "Int64",
    "site_code": "string",
    "site_name": "string",
    "source_file": "string",
})

analysis

,patient_id,visit,visit_date,sbp_mmhg,weight_kg,a1c_pct,gender,birth_year,site_code,site_name,source_file
0,P00001,baseline,2026-03-06,146,52.4,7.3,F,1966,S02,South Valley Clinic,visits_south.csv
1,P00001,week12,2026-05-27,129,50.2,NaN,F,1966,S02,South Valley Clinic,visits_south.csv
2,P00002,baseline,2026-02-20,132,85.2,8.4,F,<NA>,N01,North Clinic,visits_north.csv
3,P00002,week12,2026-05-08,129,82.0,7.4,F,<NA>,N01,North Clinic,visits_north.csv
4,P00003,baseline,2026-01-16,148,67.5,5.9,F,1970,N01,North Clinic,visits_north.csv
...,...,...,...,...,...,...,...,...,...,...,...
57176,P30038,week12,2026-04-20,135,109.0,NaN,<NA>,<NA>,<NA>,<NA>,visits_south.csv
57177,P30039,baseline,2026-02-24,142,81.8,8.0,<NA>,<NA>,<NA>,<NA>,visits_south.csv
57178,P30039,week12,2026-05-15,138,80.3,7.6,<NA>,<NA>,<NA>,<NA>,visits_south.csv
57179,P30040,baseline,2026-02-04,144,121.0,NaN,<NA>,<NA>,<NA>,<NA>,visits_south.csv


## Task 5: Validate, save, and read back

### 5.1 Validate before saving

In [48]:
# TODO: collect named True/False checks in a Series (README Task 5.1), display it, and stop with
# assert checks.all(), checks[~checks] before the analysis extract is saved.
# A failed check means an earlier step left a problem: print checks[~checks] and see HINTS.md.
checks = pd.Series({

    # analysis has the same number of rows as visits
    "analysis rows equal visits_clean": (
        len(analysis) == len(visits_clean)
    ),

    # no patient_id + visit pair repeats
    "analysis patient_id and visit pairs are unique": (
        ~analysis.duplicated(
            subset=["patient_id", "visit"]
        ).any()
    ),
    

    # every present SBP is between 70 and 250
    "sbp values are between 70 and 250": (
        analysis["sbp_mmhg"]
        .dropna()
        .between(70, 250, inclusive="both")
        .all()
    ),

    # every present site_name is a current site name
    "site names are current": (
        analysis["site_name"]
        .dropna()
        .isin(sites_current["site_name"])
        .all()
    ),


    "sbp_wide patient_id is unique": (
        sbp_wide["patient_id"].is_unique
    ),


    "raw tables equal snapshots": (
        all(
            raw[name].equals(raw_snapshot[name])
            for name in raw
        )
    ),
})


display(checks)


assert checks.all(), checks[~checks]

analysis rows equal visits_clean                  True
analysis patient_id and visit pairs are unique    True
sbp values are between 70 and 250                 True
site names are current                            True
sbp_wide patient_id is unique                     True
raw tables equal snapshots                        True
dtype: bool

### 5.2 Save the analysis extract

In [49]:
# TODO: keep the analysis rows whose patient_id is in SAMPLE_IDS, renumber them, and save them to
# ANALYSIS_SAMPLE_PATH with index=False (README Task 5.2).
analysis_sample = (
    analysis[
          analysis["patient_id"].isin(SAMPLE_IDS)
    ]
    .reset_index(drop=True)
)

analysis_sample.to_csv(
    ANALYSIS_SAMPLE_PATH,
    index=False)

analysis_sample

,patient_id,visit,visit_date,sbp_mmhg,weight_kg,a1c_pct,gender,birth_year,site_code,site_name,source_file
0,P00001,baseline,2026-03-06,146,52.4,7.3,F,1966,S02,South Valley Clinic,visits_south.csv
1,P00001,week12,2026-05-27,129,50.2,NaN,F,1966,S02,South Valley Clinic,visits_south.csv
2,P00004,baseline,2026-02-25,142,106.2,7.3,X,<NA>,N01,North Clinic,visits_north.csv
3,P00088,baseline,2026-03-10,152,68.2,8.4,F,1980,N01,North Clinic,visits_north.csv
4,P00088,week12,2026-06-03,147,66.3,7.6,F,1980,N01,North Clinic,visits_north.csv
5,P00260,baseline,NaT,70,80.4,9.5,F,1979,E04,<NA>,visits_south.csv
6,P00260,week12,2026-06-10,138,78.6,8.0,F,1979,E04,<NA>,visits_south.csv
7,P00446,baseline,NaT,250,NaN,6.5,F,1946,S02,South Valley Clinic,visits_south.csv
8,P00446,week12,2026-04-02,144,94.3,6.8,F,1946,S02,South Valley Clinic,visits_south.csv
9,P00487,baseline,2026-03-02,<NA>,103.0,7.9,M,1959,S02,South Valley Clinic,visits_south.csv


### 5.3 Read the file back

In [50]:
# TODO: read ANALYSIS_SAMPLE_PATH back with its dtypes and parse_dates=["visit_date"], and print
# whether it equals analysis_sample.
round_trip = pd.read_csv(
    ANALYSIS_SAMPLE_PATH,
    dtype={
        "patient_id": "string",
        "visit": "string",
        "sbp_mmhg": "Int64",
        "weight_kg": "float64",
        "a1c_pct": "float64",
        "gender": "string",
        "birth_year": "Int64",
        "site_code": "string",
        "site_name": "string",
        "source_file": "string",
    },
    parse_dates=["visit_date"]
)

print(round_trip.equals(analysis_sample))

True
